# E2 — Queried vs Distilled Counterfactual-Advantage Perturbation

This notebook runs the two E2 perturbation arms at each of three fixed beta values under one 2% counterfactual-label budget:

\[\text{distilled all-state perturbation} > \text{exact queried-state perturbation}.\]

The `queried` arm injects the exact teacher signal only at selected states. The `distill` arm trains a student on those same uniform labels and injects its prediction at every state. There is no uncertainty gate or active query logic; that is E3.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import make_config
from scripts.train import run_seed


## Choose one environment and a local budget

Use `taxi_cf`, `doorkey6x6_cf`, `unlockpickup_cf`, or `redbluedoors6x6_cf`. Start with a small frame budget locally; the cluster pipeline uses the YAML budget and five paired seeds.

In [ ]:
ENV_CONFIG = 'doorkey6x6_cf'  # 'taxi_cf' | 'unlockpickup_cf' | 'redbluedoors6x6_cf'
SEEDS = (0,)
FRAMES = 100_000              # local smoke budget; use None for YAML budget
LABEL_FRACTION = 0.02
BETAS = (0.25, 0.75, 1.5)     # beta=0 is the E1 PPO-GAE baseline

ARMS = {
    'queried': 'cf_queried_perturb',
    'distill': 'landscape_distill',
}


In [ ]:
def e2_config(arm: str, seed: int, beta: float):
    if arm not in ARMS:
        raise KeyError(f'unknown arm: {arm}')
    overrides = {
        'ppo.pg_mode': ARMS[arm],
        'ppo.cf_subsample': LABEL_FRACTION,
        'run.run_name': f'local_e2_{ENV_CONFIG}_{arm}_beta{beta:g}',
        'run.seeds': (seed,),
        'run.record_trajectories': False,
        'run.log_every_updates': 1,
        'distill.beta': beta,
    }
    if FRAMES is not None:
        overrides['ppo.total_timesteps'] = FRAMES
    return make_config(ENV_CONFIG, **overrides)

print(e2_config('distill', SEEDS[0], BETAS[0]).summary())


## Run paired E2 arms

`queried` uses the exact centred teacher perturbation immediately, but only at its 2% queried states. The student predicts the current rollout **before** current CF labels are added to replay; its fixed 1,024-label warm-up keeps its all-state payload at zero until it has supervision. This is not uncertainty gating.

In [ ]:
artifacts = {}
for beta in BETAS:
    for arm in ARMS:
        for seed in SEEDS:
            print(f'\n=== E2 {ENV_CONFIG}: beta {beta:g}, {arm}, seed {seed} ===')
            artifacts[(beta, arm, seed)] = run_seed(e2_config(arm, seed, beta), seed, progress=True)

artifacts


## Inspect E2-specific diagnostics

For both arms, inspect `perturb_scope_fraction` (0.02 for `queried`, 1.0 for `distill` after warm-up), `perturb_payload_rms`, `cf_n_cf_states`, `cf_branch_transitions`, and `cf_reward_coverage`. For `distill`, also inspect `distill_replay_size`, `distill_student_loss`, `distill_heldout_pg_error`, and `distill_heldout_direction_cos`.

In [ ]:
import pandas as pd

for (beta, arm, seed), item in artifacts.items():
    scalars = pd.read_csv(item['scalars_csv'])
    cols = [c for c in scalars.columns if c.startswith(('cf_', 'perturb_', 'distill_'))]
    print(f'\nbeta {beta:g}, {arm}, seed {seed}')
    display(scalars[['global_step', 'success_rate_100', *cols]].tail())
